In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Input/output paths
input_dir = "../Data/ARIMA"
output_dir = "arima_results"
os.makedirs(output_dir, exist_ok=True)
print(f"📁 Output directory '{output_dir}' is ready.\n")

# Stocks list
stocks = [f.replace(".csv", "") for f in os.listdir(input_dir) if f.endswith(".csv")]

# Store RMSE/MAE results
results = []

for stock in stocks:
    print(f"\n🔄 Processing ARIMA for {stock}...")

    # Load data
    df = pd.read_csv(f"{input_dir}/{stock}.csv", parse_dates=['Date'], index_col='Date')
    series = df.iloc[:, 0]  # assumes first column is stock price

    # Split into train/test (80/20)
    split_idx = int(len(series) * 0.8)
    train, test = series.iloc[:split_idx], series.iloc[split_idx:]

    # Plot ACF & PACF
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_acf(train, ax=axes[0])
    plot_pacf(train, ax=axes[1])
    plt.suptitle(f"{stock} - Train ACF & PACF")
    plt.savefig(f"{output_dir}/{stock}_acf_pacf.png")
    plt.close()
    print("✅ ACF & PACF plots saved.")

    # Fit ARIMA model
    model = ARIMA(train, order=(1, 1, 1))
    model_fit = model.fit()

    # Save model lightweight
    model_fit.save(f"{output_dir}/{stock}_ARIMA_model.pkl", remove_data=True)
    print(f"✅ ARIMA model for {stock} saved successfully (lightweight).")

    # Forecast
    forecast = model_fit.forecast(steps=len(test))
    forecast.index = test.index

    # Evaluate
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mae = mean_absolute_error(test, forecast)
    results.append({
        "Stock": stock,
        "RMSE": rmse,
        "MAE": mae
    })
    print(f"📉 Test RMSE: {rmse:.4f} | MAE: {mae:.4f}")

    # Plot predictions
    plt.figure(figsize=(10, 5))
    plt.plot(train.index, train, label="Train")
    plt.plot(test.index, test, label="Test Actual")
    plt.plot(test.index, forecast, label="Test Predicted")
    plt.title(f"{stock} - Actual vs Predicted (Test)")
    plt.legend()
    plt.savefig(f"{output_dir}/{stock}_prediction.png")
    plt.close()
    print("✅ Prediction plot saved.")

    # Save forecast
    forecast_df = pd.DataFrame({
        'Date': test.index,
        'Actual': test.values,
        'Predicted': forecast.values
    })
    forecast_df.to_csv(f"{output_dir}/{stock}_forecast.csv", index=False)
    print("💾 Forecast saved to CSV.")

# Save final summary
summary_df = pd.DataFrame(results)
summary_df.to_csv(f"{output_dir}/arima_summary.csv", index=False)
print("\n📊 RMSE/MAE summary saved as 'arima_summary.csv'.")
print("🎉 All stocks processed!")